In [ ]:
"""
==============================================================================
PROJETO CRÉDITO PF (BRASIL) - ETAPA 1 CONSOLIDADA MASTER
Revisão Quant Sênior: Correção de Denominador (Exposição Total), 
Malhas Geográficas Oficiais (IBGE/geobr) e Matrizes de Risco Blindadas.
Integração Completa: Geração de todos os gráficos e painéis 2x2.
Atualização: Desagregação de Regimes Macroeconômicos (Tabela 3) 
Autor: Itaiguara de Oliveira Bezerra
==============================================================================
"""

import os
import warnings
from typing import Optional

import geopandas as gpd
import geobr  # Biblioteca oficial do IBGE
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
import pandas as pd

warnings.filterwarnings("ignore")

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#4a6175"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["grid.color"] = "#dcdcdc"
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["grid.alpha"] = 0.5
pd.set_option("display.float_format", lambda x: "%.4f" % x)

def remover_molduras(ax: Optional[plt.Axes] = None) -> None:
    if ax is None:
        ax = plt.gca()
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

# ------------------------------------------------------------------------------
# 1. PARÂMETROS METODOLÓGICOS E TAXONOMIA OFICIAL (PF)
# ------------------------------------------------------------------------------
MODALIDADES_EXPURGO = [
    "Financiamentos à exportação",
    "Financiamentos à importação",
    "Financiamentos de infraestrutura e desenvolvimento",
    "Financiamentos de títulos e valores mobiliários",
    "Financiamentos com interveniência",
]

MAPA_REGIOES_DICT = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste", "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
}

NOMES_ESTADOS_BR = {
    "AC": "Acre", "AL": "Alagoas", "AP": "Amapá", "AM": "Amazonas", "BA": "Bahia", "CE": "Ceará", "DF": "Distrito Federal",
    "ES": "Espírito Santo", "GO": "Goiás", "MA": "Maranhão", "MT": "Mato Grosso", "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais", "PA": "Pará", "PB": "Paraíba", "PR": "Paraná", "PE": "Pernambuco", "PI": "Piauí",
    "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte", "RS": "Rio Grande do Sul", "RO": "Rondônia",
    "RR": "Roraima", "SC": "Santa Catarina", "SP": "São Paulo", "SE": "Sergipe", "TO": "Tocantins",
}

ORDEM_REGIOES = ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]

def mapear_macrobloco_pf_vetorizado(df: pd.DataFrame) -> np.ndarray:
    sub = df["submodalidade"].astype(str).str.lower()
    mod = df["modalidade"].astype(str).str.lower()

    cond_rotativo = sub.str.contains("crédito rotativo vinculado a cartão|rotativo|cheque especial|sem consignação|adiantamentos a depositantes|conta garantida|fatura parcelada|saque financiado|antecipação de fatura", regex=True, na=False)
    cond_consignado = sub.str.contains("com consignação|consignado", regex=True, na=False)
    cond_garantia_real = (sub.str.contains("habitacional|sfh|veículos automotores|veículos autom.|home equity|veiculo|veículo|arrendamento financeiro de veículos", regex=True, na=False) | mod.str.contains("imobiliários", regex=True, na=False))

    choices = ["Rotativo / Sem Garantia", "Consignado / Com Garantia", "Garantia Real (Imob/Auto)"]
    conditions = [cond_rotativo, cond_consignado, cond_garantia_real]
    return np.select(conditions, choices, default="Outros / Renegociados")

def estilizar_tabela_corporativa(tabela):
    for (row_idx, col_idx), cell in tabela.get_celld().items():
        cell.set_edgecolor("#dcdcdc")
        cell.set_linewidth(0.5)
        if row_idx == 0:
            cell.set_facecolor("#2c3e50")
            cell.set_text_props(color="#ffffff", weight="bold")
            cell.set_height(0.038)
        else:
            cell.set_facecolor("#f8f9fa" if row_idx % 2 == 0 else "#ffffff")
            if col_idx == 0: cell.set_text_props(ha="left")

# ------------------------------------------------------------------------------
# 2. CARREGAMENTO E SANEAMENTO DA BASE PARQUET
# ------------------------------------------------------------------------------
def main():
    print("=" * 95)
    print("ETAPA 1 (BRASIL): PROCESSAMENTO MULTIDIMENSIONAL E APÊNDICE TERRITORIAL")
    print("=" * 95)

    arquivo_parquet = "painel_scr_pf_bruto.parquet"
    if not os.path.exists(arquivo_parquet):
        raise FileNotFoundError(f"Arquivo '{arquivo_parquet}' não localizado. Conclua a Etapa 0 primeiro.")

    cols_load = [
        "data_base", "uf", "cnae_ocupacao", "porte", "modalidade", "submodalidade", 
        "carteira_ativa", "vencido_acima_de_90_dias", "ativo_problematico"
    ]

    print(f">>> Lendo colunas estratégicas de '{arquivo_parquet}'...")
    df = pd.read_parquet(arquivo_parquet, columns=cols_load)
    df["data_base"] = pd.to_datetime(df["data_base"], errors="coerce")

    df["carteira_total"] = df["carteira_ativa"].fillna(0) + df["vencido_acima_de_90_dias"].fillna(0)

    n_antes = len(df)
    df = df[~df["modalidade"].isin(MODALIDADES_EXPURGO)].copy()
    print(f"[i] Expurgo de modalidades corporativas: {n_antes - len(df):,} registros removidos.")

    df["macrobloco"] = mapear_macrobloco_pf_vetorizado(df)
    df["regiao"] = df["uf"].map(MAPA_REGIOES_DICT)

    # ------------------------------------------------------------------------------
    # 3. MOTOR DESCRITIVO ESTATÍSTICO VETORIZADO
    # ------------------------------------------------------------------------------
    def calcular_tabela_descritiva(df_base, col_dimensao, nome_dimensao):
        ts = (
            df_base.groupby(["data_base", col_dimensao])
            .agg({
                "carteira_total": "sum",
                "vencido_acima_de_90_dias": "sum",
                "ativo_problematico": "sum",
            })
            .reset_index()
        )

        ts["inad_taxa"] = (ts["vencido_acima_de_90_dias"] / ts["carteira_total"]) * 100
        ts["prob_taxa"] = (ts["ativo_problematico"] / ts["carteira_total"]) * 100

        n_datas = len(ts["data_base"].unique())
        saldo_total_medio_sistema = (df_base["carteira_total"].sum() / (n_datas * 1e9) if n_datas > 0 else 1.0)

        linhas = []
        for grupo, grp in ts.groupby(col_dimensao):
            if len(grp) >= 6:
                s_inad = grp["inad_taxa"].dropna()
                s_prob = grp["prob_taxa"].dropna()
                s_vol = (grp["carteira_total"] / 1e9).dropna()

                linhas.append({
                    nome_dimensao: str(grupo),
                    "Saldo Médio (R$ Bi)": s_vol.mean(),
                    "Part. Média (%)": (s_vol.mean() / saldo_total_medio_sistema) * 100,
                    "Inad Média (%)": s_inad.mean(),
                    "Inad Std (%)": s_inad.std(),
                    "Inad Mín (%)": s_inad.min(),
                    "Inad Mediana (%)": s_inad.median(),
                    "Inad Máx (%)": s_inad.max(),
                    "Inad Skew": s_inad.skew() if len(s_inad) >= 3 else 0.0,
                    "Inad Kurt": s_inad.kurtosis() if len(s_inad) >= 4 else 0.0,
                    "Ativo Problemático Médio (%)": s_prob.mean(),
                })
        return pd.DataFrame(linhas).sort_values("Saldo Médio (R$ Bi)", ascending=False)

    print("\n>>> Processando Tabelas Descritivas Multidimensionais...")
    tabela_macroblocos = calcular_tabela_descritiva(df, "macrobloco", "Modalidade de Crédito")
    tabela_ocupacao = calcular_tabela_descritiva(df, "cnae_ocupacao", "Perfil Ocupacional do Tomador")
    tabela_porte = calcular_tabela_descritiva(df, "porte", "Faixa de Renda do Tomador")
    tabela_regiao = calcular_tabela_descritiva(df, "regiao", "Região Geográfica")

    ts_macro = (
        df.groupby("data_base")
        .agg({
            "carteira_total": "sum",
            "vencido_acima_de_90_dias": "sum",
            "ativo_problematico": "sum",
        })
        .reset_index().sort_values("data_base")
    )
    ts_macro["taxa_inadimplencia_pct"] = (ts_macro["vencido_acima_de_90_dias"] / ts_macro["carteira_total"]) * 100
    ts_macro["taxa_problematico_pct"] = (ts_macro["ativo_problematico"] / ts_macro["carteira_total"]) * 100
    ts_macro["saldo_ativo_tri"] = ts_macro["carteira_total"] / 1e12

    # ------------------------------------------------------------------------------
    # 4. TABELAS DE CRUZAMENTO
    # ------------------------------------------------------------------------------
    print(">>> Gerando Matrizes Cruzadas: Taxas Intracélula e Distribuições Relativas...")

    cruz_ocup_total = (df.pivot_table(index="macrobloco", columns="cnae_ocupacao", values="carteira_total", aggfunc="sum") / 1e9).fillna(0.0)
    cruz_ocup_venc = (df.pivot_table(index="macrobloco", columns="cnae_ocupacao", values="vencido_acima_de_90_dias", aggfunc="sum") / 1e9).fillna(0.0)
    cruz_renda_total = (df.pivot_table(index="macrobloco", columns="porte", values="carteira_total", aggfunc="sum") / 1e9).fillna(0.0)
    cruz_renda_venc = (df.pivot_table(index="macrobloco", columns="porte", values="vencido_acima_de_90_dias", aggfunc="sum") / 1e9).fillna(0.0)

    cruz_ocup_inad = ((cruz_ocup_venc / cruz_ocup_total) * 100).fillna(0.0)
    cruz_renda_inad = ((cruz_renda_venc / cruz_renda_total) * 100).fillna(0.0)
    dist_inad_ocupacao = (cruz_ocup_venc.div(cruz_ocup_venc.sum(axis=0), axis=1) * 100).fillna(0.0)
    dist_inad_renda = (cruz_renda_venc.div(cruz_renda_venc.sum(axis=0), axis=1) * 100).fillna(0.0)

    # ------------------------------------------------------------------------------
    # 5. TABELA DO APÊNDICE: MAPEAMENTO TAXONÔMICO
    # ------------------------------------------------------------------------------
    dados_apendice_taxonomia = [
        {
            "Modalidade de Crédito": "Rotativo / Sem Garantia",
            "Denominação Formal": "Crédito Rotativo e Descoberto",
            "Submodalidades e Contratos Contábeis do SCR": (
                "Rotativo de cartão, fatura parcelada, saque financiado, antecipação de fatura, "
                "cheque especial, conta garantida, adiantamento a depositantes, crédito pessoal sem consignação"
            ),
        },
        {
            "Modalidade de Crédito": "Consignado / Com Garantia",
            "Denominação Formal": "Crédito Consignado",
            "Submodalidades e Contratos Contábeis do SCR": (
                "Crédito pessoal consignado INSS/Previdência, consignado servidor público (federal/estadual/municipal), "
                "consignado setor privado (CLT)"
            ),
        },
        {
            "Modalidade de Crédito": "Garantia Real (Imob/Auto)",
            "Denominação Formal": "Crédito com Garantia Real",
            "Submodalidades e Contratos Contábeis do SCR": (
                "Financiamento habitacional SFH e fora do SFH, CDC de veículos, arrendamento mercantil (leasing) "
                "de veículos, empréstimo com garantia de imóvel (Home Equity)"
            ),
        },
        {
            "Modalidade de Crédito": "Outros / Renegociados",
            "Denominação Formal": "Outros Créditos e Renegociações",
            "Submodalidades e Contratos Contábeis do SCR": (
                "Crédito pessoal renegociado, operações em curso anormal repactuadas, microcrédito orientado, "
                "crédito agropecuário PF, aquisição de outros bens"
            ),
        },
    ]
    df_taxonomia = pd.DataFrame(dados_apendice_taxonomia)

    # ------------------------------------------------------------------------------
    # 6. EXPORTAÇÃO EXCEL MULTI-ABA COM FORMATAÇÃO OPENPYXL
    # ------------------------------------------------------------------------------
    nome_excel_exec = "relatorio_executivo_etapa1_pf_saneado.xlsx"
    with pd.ExcelWriter(nome_excel_exec, engine="openpyxl") as writer:
        tabela_macroblocos.to_excel(writer, sheet_name="1_Macroblocos", index=False)
        tabela_ocupacao.to_excel(writer, sheet_name="2_Ocupacao_Tomador", index=False)
        tabela_porte.to_excel(writer, sheet_name="3_Faixas_Renda_Porte", index=False)
        tabela_regiao.to_excel(writer, sheet_name="4_Regioes", index=False)
        cruz_ocup_inad.reset_index().to_excel(writer, sheet_name="Cruzamento_Inad_Ocupacao", index=False)
        cruz_renda_inad.reset_index().to_excel(writer, sheet_name="Cruzamento_Inad_Renda", index=False)
        dist_inad_ocupacao.reset_index().to_excel(writer, sheet_name="Dist_Calote_Ocupacao", index=False)
        dist_inad_renda.reset_index().to_excel(writer, sheet_name="Dist_Calote_Renda", index=False)
        ts_macro.to_excel(writer, sheet_name="Serie_Macro_PF", index=False)
        df_taxonomia.to_excel(writer, sheet_name="Apendice_Taxonomia_Macroblocos", index=False)

    wb_exec = openpyxl.load_workbook(nome_excel_exec)
    header_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
    header_font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
    thin_border = Border(left=Side(style="thin", color="D9D9D9"), right=Side(style="thin", color="D9D9D9"), top=Side(style="thin", color="D9D9D9"), bottom=Side(style="thin", color="D9D9D9"))

    abas_com_percentual = ["Cruzamento_Inad_Ocupacao", "Cruzamento_Inad_Renda", "Dist_Calote_Ocupacao", "Dist_Calote_Renda"]

    for ws in wb_exec.worksheets:
        for col_idx in range(1, ws.max_column + 1):
            c_h = ws.cell(row=1, column=col_idx)
            c_h.fill = header_fill
            c_h.font = header_font
            c_h.alignment = Alignment(horizontal="center", vertical="center")
        for row_idx in range(2, ws.max_row + 1):
            for col_idx in range(1, ws.max_column + 1):
                cell = ws.cell(row=row_idx, column=col_idx)
                cell.border = thin_border
                if ws.title in abas_com_percentual and col_idx > 1:
                    cell.number_format = '0.00"%"'
                    cell.alignment = Alignment(horizontal="right", vertical="center")
    wb_exec.save(nome_excel_exec)
    print(f"[OK] Relatório executivo completo e formatado salvo: '{nome_excel_exec}'")

    # ------------------------------------------------------------------------------
    # 7. PAINEL 1: MAPA DAS MACRORREGIÕES (COM GEOBR)
    # ------------------------------------------------------------------------------
    print("\n--- GERANDO PAINEL 1: MAPA DAS MACRORREGIÕES E MACROBLOCOS PF (EDITORIAL) ---")
    try:
        resumo_reg_total = df.groupby("regiao").agg(carteira_total=("carteira_total", "sum"), vencido_total=("vencido_acima_de_90_dias", "sum")).reset_index()
        resumo_reg_total["inadimplencia_total_pct"] = (resumo_reg_total["vencido_total"] / resumo_reg_total["carteira_total"]) * 100

        resumo_reg_macro = df.groupby(["regiao", "macrobloco"]).agg(carteira_bloco=("carteira_total", "sum"), vencido_bloco=("vencido_acima_de_90_dias", "sum")).reset_index()
        resumo_reg_macro["inadimplencia_bloco_pct"] = (resumo_reg_macro["vencido_bloco"] / resumo_reg_macro["carteira_bloco"]) * 100

        gdf_estados = geobr.read_state(year=2020)
        gdf_estados = gdf_estados.rename(columns={"abbrev_state": "UF"})

        gdf_estados["regiao"] = gdf_estados["UF"].map(MAPA_REGIOES_DICT)
        gdf_regioes = gdf_estados.dissolve(by="regiao", as_index=False)
        gdf_final = gdf_regioes.merge(resumo_reg_total, on="regiao", how="left")

        fig_mapa = plt.figure(figsize=(19, 11), dpi=300)
        gs_geo = gridspec.GridSpec(5, 2, figure=fig_mapa, width_ratios=[1.15, 0.85], wspace=0.25, hspace=0.45)
        ax_mapa = fig_mapa.add_subplot(gs_geo[:, 0])

        cores_macroblocos = {"Rotativo / Sem Garantia": "#d62728", "Consignado / Com Garantia": "#2ca02c", "Garantia Real (Imob/Auto)": "#1f77b4", "Outros / Renegociados": "#ff7f0e"}

        gdf_final.plot(column="inadimplencia_total_pct", cmap="YlOrRd", linewidth=1.2, edgecolor="#4a4a4a", legend=True, legend_kwds={"label": "Taxa Agregada de Inadimplência PF (> 90 dias) %", "orientation": "horizontal", "shrink": 0.65, "pad": 0.02}, ax=ax_mapa)
        gdf_estados.plot(ax=ax_mapa, facecolor="none", edgecolor="#ffffff", linewidth=0.6, alpha=0.6)

        offset_coords = {"Norte": (-56.0, -3.5), "Nordeste": (-40.5, -8.5), "Centro-Oeste": (-54.0, -15.5), "Sudeste": (-43.5, -20.5), "Sul": (-52.0, -28.0)}

        for _, r_row in gdf_final.iterrows():
            reg_nome = str(r_row["regiao"])
            taxa_val = r_row["inadimplencia_total_pct"]
            x_c, y_c = offset_coords.get(reg_nome, (r_row.geometry.centroid.x, r_row.geometry.centroid.y))
            ax_mapa.text(x_c, y_c, f"{reg_nome}\n{taxa_val:.2f}%", ha="center", va="center", fontsize=10.5, fontweight="bold", color="#16222f", bbox=dict(boxstyle="round,pad=0.45", facecolor="#ffffff", edgecolor="#4a6175", alpha=0.92, lw=0.8))

        ax_mapa.set_axis_off()
        ax_mapa.set_title("")

        for i, reg_nome in enumerate(ORDEM_REGIOES):
            ax_sub = fig_mapa.add_subplot(gs_geo[i, 1])
            sub_bloco = resumo_reg_macro[resumo_reg_macro["regiao"] == reg_nome].sort_values("inadimplencia_bloco_pct", ascending=True)
            cores_barras = [cores_macroblocos.get(b, "#1f77b4") for b in sub_bloco["macrobloco"]]
            bars_reg = ax_sub.barh(sub_bloco["macrobloco"], sub_bloco["inadimplencia_bloco_pct"], color=cores_barras, height=0.6)
            max_v = sub_bloco["inadimplencia_bloco_pct"].max()
            for b_reg in bars_reg:
                w_reg = b_reg.get_width()
                ax_sub.text(w_reg + 0.15, b_reg.get_y() + b_reg.get_height() / 2, f"{w_reg:.2f}%", va="center", ha="left", fontsize=8.5, fontweight="bold", color="#222222")
            ax_sub.set_xlim(0, max_v * 1.25 if max_v > 0 else 1.0)
            ax_sub.set_title(f"Região {reg_nome}", fontsize=10, fontweight="bold", pad=4, color="#16222f")
            remover_molduras(ax_sub)

        nome_painel_reg = "painel_etapa1_pf_regional_macroblocos.png"
        plt.savefig(nome_painel_reg, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"[OK] Painel Regional editorial salvo: '{nome_painel_reg}'")
    except Exception as e:
        print(f"[!] Falha na etapa de Mapas Geoespaciais: {e}")

    # ------------------------------------------------------------------------------
    # 8. PAINEL 2: APÊNDICE TERRITORIAL (MAPA + TABELA ÚNICA)
    # ------------------------------------------------------------------------------
    print("\n--- GERANDO PAINEL 2: APÊNDICE TERRITORIAL (27 UFs - EDITORIAL) ---")
    try:
        df_uf_macro = df.groupby(["regiao", "uf", "macrobloco"]).agg(carteira=("carteira_total", "sum"), vencido=("vencido_acima_de_90_dias", "sum")).reset_index()
        df_uf_macro["taxa_bloco_pct"] = (df_uf_macro["vencido"] / df_uf_macro["carteira"]) * 100

        idx_max = df_uf_macro.groupby(["regiao", "uf"])["taxa_bloco_pct"].idxmax().dropna()
        tabela_critica = df_uf_macro.loc[idx_max, ["regiao", "uf", "macrobloco", "taxa_bloco_pct"]].copy()

        df_uf_geral = df.groupby("uf").agg(carteira_uf=("carteira_total", "sum"), vencido_uf=("vencido_acima_de_90_dias", "sum")).reset_index()
        df_uf_geral["taxa_uf_pct"] = (df_uf_geral["vencido_uf"] / df_uf_geral["carteira_uf"]) * 100

        df_apendice = tabela_critica.merge(df_uf_geral[["uf", "taxa_uf_pct"]], on="uf")
        df_apendice.rename(columns={"regiao": "Macrorregião", "uf": "UF", "macrobloco": "Macrobloco Crítico (Maior Risco)", "taxa_bloco_pct": "Inadimplência Macrobloco (%)", "taxa_uf_pct": "Média Geral da UF (%)"}, inplace=True)
        df_apendice["Estado"] = df_apendice["UF"].map(NOMES_ESTADOS_BR)
        df_apendice = df_apendice.sort_values("Média Geral da UF (%)", ascending=False).reset_index(drop=True)

        fig_ap = plt.figure(figsize=(18, 12), dpi=300)
        gs_ap = fig_ap.add_gridspec(1, 2, width_ratios=[1.1, 1.1], wspace=0.15, left=0.04, right=0.96)

        ax_mapa_uf = fig_ap.add_subplot(gs_ap[0, 0])
        gdf_mapa_uf = gdf_estados.merge(df_apendice[["UF", "Média Geral da UF (%)"]], on="UF", how="inner")
        gdf_mapa_uf.plot(column="Média Geral da UF (%)", cmap="YlOrRd", linewidth=0.8, ax=ax_mapa_uf, edgecolor="#4a6175", legend=True, legend_kwds={"label": "Média Geral de Inadimplência Efetiva por UF (%)", "orientation": "horizontal", "shrink": 0.70, "pad": 0.04})

        for _, row in gdf_mapa_uf.iterrows():
            pt = row.geometry.representative_point()
            ax_mapa_uf.text(pt.x, pt.y, row["UF"], fontsize=7.5, ha="center", va="center", fontweight="bold", color="#16222f")
        ax_mapa_uf.set_title("")
        ax_mapa_uf.axis("off")

        ax_tab = fig_ap.add_subplot(gs_ap[0, 1])
        ax_tab.axis("off")

        dados_tabela_unica = []
        for _, r in df_apendice.iterrows():
            dados_tabela_unica.append([r["Estado"], r["UF"], str(r["Macrobloco Crítico (Maior Risco)"]), f"{r['Inadimplência Macrobloco (%)']:.2f}%", f"{r['Média Geral da UF (%)']:.2f}%"])

        t_unica = ax_tab.table(cellText=dados_tabela_unica, colLabels=["Estado", "Sigla", "Principal Causador", "Inad. (%)", "Média (%)"], cellLoc="center", loc="center", colWidths=[0.32, 0.12, 0.26, 0.15, 0.15])
        t_unica.auto_set_font_size(False)
        t_unica.set_fontsize(8.5)
        t_unica.scale(1.0, 1.28)

        estilizar_tabela_corporativa(t_unica)
        nome_painel_apendice = "painel_apendice_pf_estados_completo.png"
        plt.savefig(nome_painel_apendice, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"[OK] Painel de Apêndice editorial salvo: '{nome_painel_apendice}'")

        nome_excel_apendice = "tabela_apendice_uf_macrobloco_critico_pf.xlsx"
        df_apendice_export = df_apendice[["Macrorregião", "UF", "Macrobloco Crítico (Maior Risco)", "Inadimplência Macrobloco (%)", "Média Geral da UF (%)"]].copy()
        df_apendice_export["Macrorregião"] = pd.Categorical(df_apendice_export["Macrorregião"], categories=ORDEM_REGIOES, ordered=True)
        df_apendice_export = df_apendice_export.sort_values(["Macrorregião", "UF"]).reset_index(drop=True)

        with pd.ExcelWriter(nome_excel_apendice, engine="openpyxl") as writer:
            df_apendice_export.to_excel(writer, sheet_name="Inadimplencia_UFs", index=False)
            df_taxonomia.to_excel(writer, sheet_name="Dicionario_Macroblocos", index=False)

        wb = openpyxl.load_workbook(nome_excel_apendice)
        ws = wb["Inadimplencia_UFs"]
        bold_font = Font(name="Arial", size=10, bold=True)
        center_align = Alignment(horizontal="center", vertical="center")

        for col in range(1, 6):
            c_h = ws.cell(row=1, column=col)
            c_h.fill = header_fill
            c_h.font = header_font
            c_h.alignment = center_align

        linhas_por_regiao = df_apendice_export["Macrorregião"].value_counts()[ORDEM_REGIOES]
        linha_atual = 2
        for reg, count in linhas_por_regiao.items():
            if count > 0:
                linha_fim = linha_atual + count - 1
                if linha_fim > linha_atual: ws.merge_cells(start_row=linha_atual, start_column=1, end_row=linha_fim, end_column=1)
                cell_reg = ws.cell(row=linha_atual, column=1)
                cell_reg.alignment = center_align
                cell_reg.font = bold_font
                for r in range(linha_atual, linha_fim + 1):
                    for c in range(1, 6):
                        cell_body = ws.cell(row=r, column=c)
                        cell_body.border = thin_border
                        if c in [4, 5]: cell_body.number_format = '0.00"%\"'; cell_body.alignment = Alignment(horizontal="right", vertical="center")
                        elif c in [1, 2]: cell_body.alignment = center_align
                        else: cell_body.alignment = Alignment(horizontal="left", vertical="center")
                linha_atual = linha_fim + 1

        ws.column_dimensions["A"].width = 16
        ws.column_dimensions["B"].width = 8
        ws.column_dimensions["C"].width = 30
        ws.column_dimensions["D"].width = 28
        ws.column_dimensions["E"].width = 22

        ws_tax = wb["Dicionario_Macroblocos"]
        for col in range(1, 4):
            c_h = ws_tax.cell(row=1, column=col); c_h.fill = header_fill; c_h.font = header_font; c_h.alignment = center_align
        for r in range(2, ws_tax.max_row + 1):
            for c in range(1, 4):
                cell_body = ws_tax.cell(row=r, column=c); cell_body.border = thin_border; cell_body.font = Font(name="Arial", size=9.5)
                if c == 1: cell_body.font = Font(name="Arial", size=9.5, bold=True); cell_body.alignment = Alignment(horizontal="left", vertical="center")
                elif c == 2: cell_body.alignment = Alignment(horizontal="left", vertical="center")
                else: cell_body.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
        ws_tax.column_dimensions["A"].width = 28; ws_tax.column_dimensions["B"].width = 32; ws_tax.column_dimensions["C"].width = 95
        wb.save(nome_excel_apendice)
        print(f"[OK] Planilha '{nome_excel_apendice}' salva com formatação executiva!")
    except Exception as e:
        print(f"[!] Falha na etapa do Apêndice Territorial: {e}")


    # ------------------------------------------------------------------------------
    # 10. PAINÉIS 2x2: EVOLUÇÃO DE SALDOS E DISPERSÃO EDA (COM TÍTULOS NOS SUBPLOTS)
    # ------------------------------------------------------------------------------
    print("\n--- GERANDO PAINÉIS 2x2: SALDOS HISTÓRICOS E DISPERSÃO EDA (EDITORIAL) ---")

    ts_saldos = df.groupby(["data_base", "macrobloco"])["carteira_total"].sum().reset_index()
    ts_saldos["saldo_tri"] = ts_saldos["carteira_total"] / 1e12

    macroblocos_ordem = [("Rotativo / Sem Garantia", "#d62728"), ("Consignado / Com Garantia", "#1f77b4"), ("Garantia Real (Imob/Auto)", "#2ca02c"), ("Outros / Renegociados", "#ff7f0e")]

    fig_saldo, axes_saldo = plt.subplots(2, 2, figsize=(14, 9), dpi=300)
    for idx, (mb, cor) in enumerate(macroblocos_ordem):
        ax = axes_saldo.flatten()[idx]
        df_mb = ts_saldos[ts_saldos["macrobloco"] == mb].sort_values("data_base")
        ax.plot(df_mb["data_base"], df_mb["saldo_tri"], color=cor, lw=2.2)
        ax.fill_between(df_mb["data_base"], df_mb["saldo_tri"], color=cor, alpha=0.1)
        ax.axvline(pd.to_datetime("2020-01-01"), color="gray", linestyle=":", lw=1.2, alpha=0.8)
        ax.set_title(mb, fontsize=11, fontweight="bold", pad=8, color="#16222f")
        ax.set_xlabel("Ano", fontsize=9.5, fontweight="bold")
        ax.set_ylabel("Saldo da Exposição Total (R$ Trilhões)", fontsize=9.5, fontweight="bold")
        ax.grid(True)
        ymax = df_mb["saldo_tri"].max()
        ax.set_ylim(0, ymax * 1.15 if ymax > 0 else 1)
        remover_molduras(ax)

    plt.tight_layout()
    nome_painel_saldos = "painel_etapa1_pf_evolucao_saldos_4macroblocos.png"
    plt.savefig(nome_painel_saldos, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[OK] Painel de Saldos editorial salvo: '{nome_painel_saldos}'")

    fig_eda, axes_eda = plt.subplots(2, 2, figsize=(15, 11), dpi=300)

    ts_macro_b = df.groupby(["data_base", "macrobloco"]).agg({"carteira_total": "sum", "vencido_acima_de_90_dias": "sum"}).reset_index()
    ts_macro_b["inad_pct"] = (ts_macro_b["vencido_acima_de_90_dias"] / ts_macro_b["carteira_total"]) * 100
    for mb, grp in ts_macro_b.groupby("macrobloco"):
        axes_eda[0, 0].plot(grp["data_base"], grp["inad_pct"], lw=2, label=mb)
    axes_eda[0, 0].set_title("Inadimplência por Linha de Crédito", fontsize=10.5, fontweight="bold", pad=8, color="#16222f")
    axes_eda[0, 0].set_xlabel("Ano", fontsize=9.5, fontweight="bold"); axes_eda[0, 0].set_ylabel("Taxa de Inadimplência (%)", fontsize=10, fontweight="bold")
    axes_eda[0, 0].grid(True); axes_eda[0, 0].legend(frameon=False, facecolor="#f8f9fa", fontsize=8.5); remover_molduras(axes_eda[0, 0])

    ts_ocupa = df.groupby(["data_base", "cnae_ocupacao"]).agg({"carteira_total": "sum", "vencido_acima_de_90_dias": "sum"}).reset_index()
    ts_ocupa["inad_pct"] = (ts_ocupa["vencido_acima_de_90_dias"] / ts_ocupa["carteira_total"]) * 100
    for oc in ["Aposentado/pensionista", "Servidor ou empregado público", "Empregado de empresa privada", "Autônomo", "MEI"]:
        grp = ts_ocupa[ts_ocupa["cnae_ocupacao"] == oc]
        if not grp.empty: axes_eda[0, 1].plot(grp["data_base"], grp["inad_pct"], lw=1.8, label=oc)
    axes_eda[0, 1].set_title("Inadimplência por Perfil Ocupacional", fontsize=10.5, fontweight="bold", pad=8, color="#16222f")
    axes_eda[0, 1].set_xlabel("Ano", fontsize=9.5, fontweight="bold"); axes_eda[0, 1].set_ylabel("Taxa de Inadimplência (%)", fontsize=10, fontweight="bold")
    axes_eda[0, 1].grid(True); axes_eda[0, 1].legend(frameon=False, facecolor="#f8f9fa", fontsize=8.5); remover_molduras(axes_eda[0, 1])

    axes_eda[1, 0].scatter(ts_macro["taxa_inadimplencia_pct"], ts_macro["taxa_problematico_pct"], color="#2b5c8f", s=45, alpha=0.7)
    axes_eda[1, 0].set_title("Dispersão: Inadimplência (>90d) vs. Ativo Problemático Total", fontsize=10.5, fontweight="bold", pad=8, color="#16222f")
    axes_eda[1, 0].set_xlabel("Inadimplência > 90 dias (%)", fontsize=10, fontweight="bold"); axes_eda[1, 0].set_ylabel("Ativo Problemático Total (%)", fontsize=10, fontweight="bold")
    axes_eda[1, 0].grid(True); remover_molduras(axes_eda[1, 0])

    ts_porte = df.groupby(["data_base", "porte"]).agg({"carteira_total": "sum", "vencido_acima_de_90_dias": "sum"}).reset_index()
    ts_porte["inad_pct"] = (ts_porte["vencido_acima_de_90_dias"] / ts_porte["carteira_total"]) * 100
    for pt in ["Até 1 salário mínimo", "Mais de 1 a 2 salários mínimos", "Mais de 2 a 3 salários mínimos", "Mais de 5 a 10 salários mínimos", "Acima de 20 salários mínimos"]:
        grp = ts_porte[ts_porte["porte"] == pt]
        if not grp.empty: axes_eda[1, 1].plot(grp["data_base"], grp["inad_pct"], lw=1.8, label=pt)
    axes_eda[1, 1].set_title("Inadimplência por Faixa de Renda do Tomador", fontsize=10.5, fontweight="bold", pad=8, color="#16222f")
    axes_eda[1, 1].set_xlabel("Ano", fontsize=9.5, fontweight="bold"); axes_eda[1, 1].set_ylabel("Taxa de Inadimplência (%)", fontsize=10, fontweight="bold")
    axes_eda[1, 1].grid(True); axes_eda[1, 1].legend(frameon=False, facecolor="#f8f9fa", fontsize=8.5); remover_molduras(axes_eda[1, 1])

    plt.tight_layout()
    nome_painel_eda = "painel_eda_etapa1_pf_multidimensional.png"
    plt.savefig(nome_painel_eda, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[OK] Painel EDA Multidimensional editorial salvo: '{nome_painel_eda}'")

    # ------------------------------------------------------------------------------
    # 11. IMPACTO DOS REGIMES MACROECONÔMICOS NOS SALDOS (REVISADO)
    # ------------------------------------------------------------------------------
    print("\n--- IMPACTO DOS REGIMES MACROECONÔMICOS NOS SALDOS (Tabela 3 Revisada) ---")
    ts_saldos_bi = ts_saldos.copy()
    ts_saldos_bi["saldo_bi"] = ts_saldos_bi["carteira_total"] / 1e9

    # 1. Pré-Choque: até Fev/2020
    pre_choque = ts_saldos_bi[ts_saldos_bi["data_base"] <= '2020-02-29'].groupby('macrobloco')['saldo_bi'].mean()
    
    # 2. Injeção de Liquidez/Estímulo: Mar/2020 a Dez/2021
    estimulo = ts_saldos_bi[(ts_saldos_bi["data_base"] >= '2020-03-01') & (ts_saldos_bi["data_base"] <= '2021-12-31')].groupby('macrobloco')['saldo_bi'].mean()
    
    # 3. Aperto Monetário: Jan/2022 a Dez/2023
    aperto = ts_saldos_bi[(ts_saldos_bi["data_base"] >= '2022-01-01') & (ts_saldos_bi["data_base"] <= '2023-12-31')].groupby('macrobloco')['saldo_bi'].mean()
    
    # 4. Normalização: Jan/2024 em diante
    normalizacao = ts_saldos_bi[ts_saldos_bi["data_base"] >= '2024-01-01'].groupby('macrobloco')['saldo_bi'].mean()

    # Cálculo das variações sequenciais
    var_estimulo = ((estimulo - pre_choque) / pre_choque) * 100
    var_aperto = ((aperto - estimulo) / estimulo) * 100
    var_normalizacao = ((normalizacao - aperto) / aperto) * 100

    resultado = pd.DataFrame({
        '1. Pré-Choque (R$ Bi)': pre_choque, 
        '2. Estímulo (R$ Bi)': estimulo, 
        'Var. Estímulo (%)': var_estimulo,
        '3. Aperto (R$ Bi)': aperto,
        'Var. Aperto (%)': var_aperto,
        '4. Normalização (R$ Bi)': normalizacao,
        'Var. Normalização (%)': var_normalizacao
    }).round(2)

    print(resultado)

    print("\n" + "=" * 95)
    print("[OK] Execução e salvamento concluídos com sucesso e regimes macroeconômicos particionados.")
    print("=" * 95)

if __name__ == '__main__':
    main()